# Generating Python Code with Granite Code Models

**NOTE:** This recipe demonstrates how to use Granite Code Models to generate Python code from text prompts.

## Setting Up

### Install dependencies

In [ ]:
!pip install git+https://github.com/ibm-granite-community/utils \
    langchain_community \
    replicate

## Select a model

In [ ]:
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import get_env_var

model = Replicate(
    model="ibm-granite/granite-8b-code-instruct-128k",
    replicate_api_token=get_env_var('REPLICATE_API_TOKEN'),
)

### What is Fill-in-the-Middle (FIM)?

Fill-in-the-Middle (FIM) is a technique where the model is given both the beginning and end portions of code, and is asked to generate the middle part. This approach is particularly useful for:

1. Fixing bugs in existing code
2. Adding missing implementation details
3. Translating code between programming languages
4. Generating unit tests
5. Refactoring and improving code quality
6. Adding security improvements to vulnerable code

### Few-shot Fill-in-the-Middle Prompting with Granite Code Models

In few-shot prompting, you provide the model with a question and some examples to help it understand the pattern you want.

**Provide a list of Q&A examples**

In [ ]:
examples = [
    {
        "question": """
            Given a buggy code snippet and a partially written function, fill in the correct logic to fix the bug:

            def validate_email(email):
                if '@' not in email or '.' not in email.split('@')[-1]:
                    return False
                return True
        """,
        "answer": """
        def validate_email(email):
            import re
            pattern = r"^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"
            return re.match(pattern, email) is not None
        """,
    },
    {
        "question": """
            Translate the following Python code to JavaScript:

            def add(a, b):
                return a + b
        """,
        "answer": """
        function add(a, b) {
            return a + b;
        }
        """,
    },
    {
        "question": """
            Automatically generate unit tests for the given function:

            def calculate_discount(price, discount):
                discounted_price = price - (price * discount)
                return max(discounted_price, 0)
        """,
        "answer": """
        def calculate_discount(price, discount):
            discounted_price = price - (price * discount)
            return max(discounted_price, 0)

        import unittest

        class TestCalculateDiscount(unittest.TestCase):
            def test_normal_case(self):
                self.assertEqual(calculate_discount(100, 0.2), 80)

            def test_zero_discount(self):
                self.assertEqual(calculate_discount(100, 0.0), 100)

            def test_full_discount(self):
                self.assertEqual(calculate_discount(100, 1.0), 0)

            def test_negative_result(self):
                self.assertEqual(calculate_discount(50, 2.0), 0)
        """,
    },
    {
        "question": """
            Given a long function, propose modular refactoring to fill in refactored helper methods:

            def process_data(data):
                clean_data = [d.strip().lower() for d in data if d]
                sorted_data = sorted(clean_data)
                unique_data = list(set(sorted_data))
                return unique_data
        """,
        "answer": """
        def clean(data):
            return [d.strip().lower() for d in data if d]

        def process_data(data):
            cleaned = clean(data)
            sorted_data = sorted(cleaned)
            return list(set(sorted_data))
        """,
    },
    {
        "question": """
            Automatically insert security best practices into this insecure code snippet:

            def connect_to_db(user_input):
                query = "SELECT * FROM users WHERE name = '" + user_input + "'"
                cursor.execute(query)
        """,
        "answer": """
        def connect_to_db(user_input):
            query = "SELECT * FROM users WHERE name = %s"
            cursor.execute(query, (user_input,))
        """,
    },
    {
        "question": """
            Suggest missing preprocessing steps in this ML pipeline:

            def preprocess_data(df):
                df = df.dropna()
                df = df[df['age'] > 0]
                return df
        """,
        "answer": """
        def preprocess_data(df):
            df = df.dropna()
            df = df[df['age'] > 0]
            df = df.drop_duplicates()
            df['income'] = df['income'].fillna(df['income'].median())
            df = df.reset_index(drop=True)
            return df
        """,
    },
    {
        "question": """
            Add missing import statements:

            def parse_json(data):
                return json.loads(data)
        """,
        "answer": """
        import json

        def parse_json(data):
            return json.loads(data)
        """,
    },
    {
        "question": """
            Insert inline comments explaining logic line-by-line:

            def fibonacci(n):
                if n <= 1:
                    return n
                return fibonacci(n-1) + fibonacci(n-2)
        """,
        "answer": """
        def fibonacci(n):
            if n <= 1:  # Base case: return n if it's 0 or 1
                return n
            return fibonacci(n-1) + fibonacci(n-2)  # Recursive call: sum of two previous terms
        """,
    }
]

## Assemble the prompt template

Few-shot prompt + system prompt + user prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.messages import SystemMessage
from textwrap import dedent

system_prompt = SystemMessage(content=dedent("""\
    You are a Python code generation specialist. Your task is to perform Fill-in-the-Middle (FIM)
    code completion following these strict guidelines:

    1. Generate ONLY executable Python code without any explanations or comments outside of docstrings
    2. Follow PEP 8 style guidelines and best practices
    3. Include appropriate type hints for all function parameters and return values
    4. Write comprehensive docstrings following Google docstring format
    5. Ensure the generated code fits seamlessly between the provided prefix and suffix
    6. Maintain consistent indentation and naming conventions with surrounding code
    7. Optimize for readability, efficiency, and maintainability
    8. Do not include any natural language responses or explanations outside of code

    Your response must consist solely of valid Python code that can be directly executed.
"""))

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

chat_template_with_system_message = ChatPromptTemplate.from_messages(
    [
        system_prompt,
        few_shot_prompt,
        ("human", "{question}"),
    ]
)

print(chat_template_with_system_message.input_variables)

## Create a chat-model chain

In [ ]:
chain_sys = chat_template_with_system_message | model

### Example №1

In [ ]:
prompt = """
    Given a buggy code snippet and a partially written function, fill in the correct logic to fix the bug:

    def is_palindrome(s):
        if s == s[::-1]:
            return True
        return False
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

def is_palindrome(s):
    s = ''.join(char.lower() for char in s if char.isalnum())
    return s == s[::-1]

print(is_palindrome("Was it a car or a cat I saw?"))  # Should return True

### Example №2


In [ ]:
prompt = """
    Translate the following Python code to Java:

    def find_max(numbers):
        return max(numbers)
"""

response = chain_sys.invoke({"question": prompt})
print(response)

```java
public int findMax(int[] numbers) {
    return Arrays.stream(numbers).max().getAsInt();
}
```

### Example №3


In [ ]:
prompt = """
    Automatically generate unit tests for the given function:

    def factorial(n):
        if n <= 1:
            return 1
        return n * factorial(n - 1)
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

import unittest

def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

class TestFactorial(unittest.TestCase):
    def test_normal_case(self):
        self.assertEqual(factorial(5), 120)

    def test_edge_case(self):
        self.assertEqual(factorial(0), 1)

    def test_negative_case(self):
        with self.assertRaises(ValueError):
            factorial(-1)

if __name__ == '__main__':
    unittest.main()

### Example №4


In [ ]:
prompt = """
    Automatically insert security best practices into this insecure code snippet:

    def hash_password(password):
        import hashlib
        return hashlib.md5(password.encode()).hexdigest()
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

def hash_password(password):
    import hashlib
    import secrets
    salt = secrets.token_hex(16)
    return hashlib.pbkdf2_hmac('sha256', password.encode(), salt.encode(), 100000).hex() + ':' + salt

### Example №5


In [ ]:
prompt = """
    Suggest missing preprocessing steps in this ML pipeline using FIM:

    def preprocess_text_data(df):
        df['text'] = df['text'].str.lower()
        df = df[df['text'].str.len() > 0]
        return df
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

def preprocess_text_data(df):
    df['text'] = df['text'].str.lower()
    df = df[df['text'].str.len() > 0]
    df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)
    df['text'] = df['text'].str.strip()
    return df

### Example №6


In [ ]:
prompt = """
    Add missing import statements using code completion:

    def current_timestamp():
        return datetime.now().isoformat()

    def parse_date(date_string):
        return datetime.strptime(date_string, '%Y-%m-%d')
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

from datetime import datetime

def current_timestamp():
    return datetime.now().isoformat()

def parse_date(date_string):
    return datetime.strptime(date_string, '%Y-%m-%d')

### Example №7


In [ ]:
prompt = """
    Insert inline comments explaining logic line-by-line:

    def binary_search(arr, target):
        left, right = 0, len(arr) - 1
        while left <= right:
            mid = (left + right) // 2
            if arr[mid] == target:
                return mid
            elif arr[mid] < target:
                left = mid + 1
            else:
                right = mid - 1
        return -1
"""

response = chain_sys.invoke({"question": prompt})
print(response)

In [ ]:
# Paste your code here to test it:

def binary_search(arr, target):
    left, right = 0, len(arr) - 1  # Initialize search boundaries
    while left <= right:  # Continue while search space exists
        mid = (left + right) // 2  # Calculate middle index
        if arr[mid] == target:  # Found target value
            return mid
        elif arr[mid] < target:  # Target is in right half
            left = mid + 1
        else:  # Target is in left half
            right = mid - 1
    return -1  # Target not found